In [6]:
--1 Get the total number of orders per customer. Uses row number to get the order number and then count to displat the total number of orders
SELECT 
    CustomerID,
    SalesOrderID,
    OrderDate,
    ROW_NUMBER() OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS OrderNumber,
    COUNT(*) OVER (PARTITION BY CustomerID) AS TotalOrders
FROM Sales.SalesOrderHeader;

(31465 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.365

CustomerID,SalesOrderID,OrderDate,OrderNumber,TotalOrders
11000,43793,2011-06-21 00:00:00.000,1,3
11000,51522,2013-06-20 00:00:00.000,2,3
11000,57418,2013-10-03 00:00:00.000,3,3
11001,43767,2011-06-17 00:00:00.000,1,3
11001,51493,2013-06-18 00:00:00.000,2,3
11001,72773,2014-05-12 00:00:00.000,3,3
11002,43736,2011-06-09 00:00:00.000,1,3
11002,51238,2013-06-02 00:00:00.000,2,3
11002,53237,2013-07-26 00:00:00.000,3,3
11003,43701,2011-05-31 00:00:00.000,1,3


In [10]:
--2 Get the first order date for each customer. Used first value to get the first order date
SELECT 
    CustomerID,
    OrderDate,
    FIRST_VALUE(OrderDate) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS FirstOrderDate
FROM Sales.SalesOrderHeader;

(31465 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.599

CustomerID,OrderDate,FirstOrderDate
11000,2011-06-21 00:00:00.000,2011-06-21 00:00:00.000
11000,2013-06-20 00:00:00.000,2011-06-21 00:00:00.000
11000,2013-10-03 00:00:00.000,2011-06-21 00:00:00.000
11001,2011-06-17 00:00:00.000,2011-06-17 00:00:00.000
11001,2013-06-18 00:00:00.000,2011-06-17 00:00:00.000
11001,2014-05-12 00:00:00.000,2011-06-17 00:00:00.000
11002,2011-06-09 00:00:00.000,2011-06-09 00:00:00.000
11002,2013-06-02 00:00:00.000,2011-06-09 00:00:00.000
11002,2013-07-26 00:00:00.000,2011-06-09 00:00:00.000
11003,2011-05-31 00:00:00.000,2011-05-31 00:00:00.000


In [11]:
--3 Get a running total of sales made by each sales person. Sum the total due to get the running total
SELECT 
    p.BusinessEntityID,
    o.SalesOrderID,
    o.OrderDate,
    o.TotalDue,
    SUM(o.TotalDue) OVER (PARTITION BY p.BusinessEntityID ORDER BY o.OrderDate) AS RunningTotal
FROM Sales.SalesOrderHeader AS o
JOIN Sales.SalesPerson p ON o.SalesPersonID = p.BusinessEntityID;

(3806 rows affected)

Total execution time: 00:00:00.493

BusinessEntityID,SalesOrderID,OrderDate,TotalDue,RunningTotal
274,43849,2011-07-01 00:00:00.000,23130.2957,23130.2957
274,44082,2011-08-01 00:00:00.000,2297.0332,25427.3289
274,44508,2011-10-01 00:00:00.000,4723.1073,32567.9155
274,44532,2011-10-01 00:00:00.000,2417.4793,32567.9155
274,45317,2012-01-01 00:00:00.000,68918.2404,101486.1559
274,45526,2012-01-29 00:00:00.000,6870.0165,122100.3986
274,45546,2012-01-29 00:00:00.000,13744.2262,122100.3986
274,45814,2012-02-29 00:00:00.000,37625.4303,159725.8289
274,46334,2012-04-30 00:00:00.000,51204.0606,215389.9631
274,46369,2012-04-30 00:00:00.000,4460.0736,215389.9631


In [12]:
--4 Get previous order date for each customer. Use lag to get the prior date
SELECT 
    CustomerID,
    SalesOrderID,
    OrderDate,
    LAG(OrderDate) OVER (PARTITION BY CustomerID ORDER BY OrderDate) AS PreviousOrderDate
FROM Sales.SalesOrderHeader;

(31465 rows affected)

Displaying Top 5000 rows.

Total execution time: 00:00:00.959

CustomerID,SalesOrderID,OrderDate,PreviousOrderDate
11000,43793,2011-06-21 00:00:00.000,NULL
11000,51522,2013-06-20 00:00:00.000,2011-06-21 00:00:00.000
11000,57418,2013-10-03 00:00:00.000,2013-06-20 00:00:00.000
11001,43767,2011-06-17 00:00:00.000,NULL
11001,51493,2013-06-18 00:00:00.000,2011-06-17 00:00:00.000
11001,72773,2014-05-12 00:00:00.000,2013-06-18 00:00:00.000
11002,43736,2011-06-09 00:00:00.000,NULL
11002,51238,2013-06-02 00:00:00.000,2011-06-09 00:00:00.000
11002,53237,2013-07-26 00:00:00.000,2013-06-02 00:00:00.000
11003,43701,2011-05-31 00:00:00.000,NULL


In [13]:
--5 Get top 5 employees based on number of sales made. CTE to first get ranking, sum to get total amount made in sales, and then row number to rank
WITH TopSales AS (
    SELECT 
        SalesPersonID,
        SUM(TotalDue) AS TotalSales,
        ROW_NUMBER() OVER (ORDER BY SUM(TotalDue) DESC) AS Rank
    FROM Sales.SalesOrderHeader
    WHERE SalesPersonID IS NOT NULL
    GROUP BY SalesPersonID
)
SELECT 
    SalesPersonID,
    TotalSales,
    Rank
FROM TopSales
WHERE Rank <= 5;

(5 rows affected)

Total execution time: 00:00:00.238

SalesPersonID,TotalSales,Rank
276,11695019.0605,1
277,11342385.8968,2
275,10475367.0751,3
289,9585124.9477,4
279,8086073.6761,5


In [14]:
--6 Get top 3 customers based on how much they spent on purchases. CTE to first get ranking, sum to get total amount spent, and then row number to rank
WITH RankedCustomers AS (
    SELECT 
        CustomerID,
        SUM(TotalDue) AS TotalSpent,
        ROW_NUMBER() OVER (ORDER BY SUM(TotalDue) DESC) AS Rank
    FROM Sales.SalesOrderHeader
    GROUP BY CustomerID
)
SELECT 
    CustomerID,
    TotalSpent,
    Rank
FROM RankedCustomers
WHERE Rank <= 3;

(3 rows affected)

Total execution time: 00:00:00.158

CustomerID,TotalSpent,Rank
29818,989184.082,1
29715,961675.8596,2
29722,954021.9235,3


In [17]:
--7 Get which products are sold more often. Sum to get the quantity of the product to see how many times it was bought. Row number to rank which products were sold more 
SELECT 
    p.ProductID,
    p.Name,
    SUM(o.OrderQty) AS QuantitySold,
    ROW_NUMBER() OVER (ORDER BY SUM(o.OrderQty) DESC) AS Rank
FROM Sales.SalesOrderDetail o
JOIN Production.Product p ON o.ProductID = p.ProductID
GROUP BY p.ProductID, p.Name
ORDER BY Rank;

(266 rows affected)

Total execution time: 00:00:00.129

ProductID,Name,QuantitySold,Rank
712,AWC Logo Cap,8311,1
870,Water Bottle - 30 oz.,6815,2
711,"Sport-100 Helmet, Blue",6743,3
715,"Long-Sleeve Logo Jersey, L",6592,4
708,"Sport-100 Helmet, Black",6532,5
707,"Sport-100 Helmet, Red",6266,6
864,"Classic Vest, S",4247,7
873,Patch Kit/8 Patches,3865,8
884,"Short-Sleeve Classic Jersey, XL",3864,9
714,"Long-Sleeve Logo Jersey, M",3636,10


In [19]:
--8 Get which month had the most revenue. Sum totaldue for each month to get how much profit was made and rank to get which month had more profit
SELECT 
    MONTH(OrderDate) AS SalesMonth,
    YEAR(OrderDate) AS SalesYear,
    SUM(TotalDue) AS TotalSales,
    RANK() OVER (ORDER BY SUM(TotalDue) DESC) AS Rank
FROM Sales.SalesOrderHeader
GROUP BY MONTH(OrderDate), YEAR(OrderDate)
ORDER BY Rank;

(38 rows affected)

Total execution time: 00:00:00.086

SalesMonth,SalesYear,TotalSales,Rank
3,2014,8097036.3137,1
5,2014,6006183.211,2
6,2013,5726265.2635,3
7,2013,5521840.8445,4
10,2013,5374375.9418,5
10,2011,5156269.5291,6
9,2013,5083505.3374,7
1,2014,4798027.8709,8
6,2012,4610647.2153,9
12,2013,4560577.0958,10


In [21]:
--9 Get the first and last date that each product was sold. use first value and last value 
SELECT DISTINCT
    ProductID,
    FIRST_VALUE(OrderDate) OVER (PARTITION BY ProductID ORDER BY OrderDate) AS FirstDate,
    LAST_VALUE(OrderDate) OVER (
        PARTITION BY ProductID 
        ORDER BY OrderDate 
        ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING) AS LastDate
FROM Sales.SalesOrderDetail AS d
JOIN Sales.SalesOrderHeader o ON d.SalesOrderID = o.SalesOrderID;

(266 rows affected)

Total execution time: 00:00:01.597

ProductID,FirstDate,LastDate
916,2013-05-30 00:00:00.000,2014-05-01 00:00:00.000
992,2013-05-30 00:00:00.000,2014-05-28 00:00:00.000
869,2013-05-30 00:00:00.000,2014-06-30 00:00:00.000
841,2012-05-30 00:00:00.000,2013-04-30 00:00:00.000
765,2011-05-31 00:00:00.000,2014-03-30 00:00:00.000
709,2011-05-31 00:00:00.000,2012-04-30 00:00:00.000
970,2013-05-30 00:00:00.000,2014-05-29 00:00:00.000
758,2011-05-31 00:00:00.000,2012-04-30 00:00:00.000
902,2013-05-30 00:00:00.000,2013-07-31 00:00:00.000
903,2013-05-30 00:00:00.000,2013-07-31 00:00:00.000


In [22]:
--10 Get how many times a product has been sold compared to the total number of sales. Use count to get how many times it was sold and sum all to et total sales
SELECT 
    ProductID,
    COUNT(*) AS TimesSold,
    SUM(COUNT(*)) OVER () AS TotalSales
FROM Sales.SalesOrderDetail
GROUP BY ProductID;

(266 rows affected)

Total execution time: 00:00:00.099

ProductID,TimesSold,TotalSales
707,3083,121317
708,3007,121317
709,188,121317
710,44,121317
711,3090,121317
712,3382,121317
713,429,121317
714,1218,121317
715,1635,121317
716,1076,121317
